In [15]:
library(survival)
library(dplyr)
library(tidyr)
library(readr)
library(stringr)
library(tibble)


In [18]:
MIN_PATIENTS <- 10

# Parse arguments
args <- commandArgs(trailingOnly = TRUE)

# !!!!!!!!!!
#tissuefile <- args[1]
#output_dir <- args[2]

tissuefile <- "/home/lnemati/pathway_crosstalk/data/survival_data/tissues/lung.csv"

# Read the survival data
cat('Reading survival data\n')
df <- read_csv(tissuefile, col_types = cols())

# Check and fix column names if needed
if (names(df)[1] == "") {
  names(df)[1] <- "sample"  # Rename if the first column has no name
}

# Set the first column as row names
df <- df %>% column_to_rownames(var = names(df)[1])

# The ids are like this TCGA-3M-AB46-01, the last number after last - represents samples, remove it to get patient id
df$patient <- sapply(strsplit(rownames(df), '-'), function(x) paste(x[1:4], collapse = '-'))

# Print number of patients (rows) in the dataset
cat('Number of patients:', nrow(df), '\n')
                 
covariates_dir = '/home/lnemati/pathway_crosstalk/results/survival/covariates/'
                         
# Find matching covariates file
all_covariates_files <- list.files(covariates_dir, full.names = TRUE)
tissue_name <- tools::file_path_sans_ext(basename(tissuefile))
tissue_name_pattern <- gsub(" ", "_", tissue_name, fixed = TRUE)

matching_file <- all_covariates_files[
    grepl(tissue_name_pattern, all_covariates_files, ignore.case = TRUE)
]

if (length(matching_file) == 0) {
    cat("No matching covariates file found for tissue:", tissue_name, "\n") 
    cat("Done: covariates_survival.R - no covariates file")
    stop("Done: covariates_survival.R - no covariates file")
} else if (length(matching_file) > 1) {
    cat("Multiple matching covariates files found for tissue:", tissue_name, "\n")
    cat("Done: covariates_survival.R - multiple covariates files")
    stop("Done: covariates_survival.R - multiple covariates files")
}

cat('Reading covariates from:', matching_file, '\n')
covariates_df <- read_csv(matching_file, col_types = cols())
circuits <- covariates_df %>% pull(interaction)             
                     
#motifs_df <- read_csv('/home/lnemati/pathway_crosstalk/results/survival/covariates/Brain.csv') # Only better only lungs
#circuits <- motifs_df %>% pull(interaction)
#cliques <- motifs_df %>%
#  filter(motif %in% c("3_clique", "4_clique")) %>%
#  pull(interaction)
                     
# Extract ccc interactions from circuits by splitting and flattening the list
ccc <- unique(unlist(str_split(circuits, '&')))  # Split interactions by '&' and flatten the list
cat('Number of ccc interactions:', length(ccc), '\n')

multiple <- length(unique(df$condition)) > 1

if (multiple) {
  df <- df %>%
    group_by(condition) %>%
    mutate(
      binary_age = as.integer(age_at_diagnosis > median(age_at_diagnosis, na.rm = TRUE)),
      gender = as.integer(gender == "Female")
    ) %>%
    ungroup()
} else {
  df <- df %>%
    mutate(
      binary_age = as.integer(age_at_diagnosis > median(age_at_diagnosis, na.rm = TRUE)),
      gender = as.integer(gender == "Female")
    )
}

Reading survival data


New names:
• `` -> `...1`


Number of patients: 1122 
Reading covariates from: /home/lnemati/pathway_crosstalk/results/survival/covariates//Lung.csv 
Number of ccc interactions: 282 


In [26]:
survival_analysis <- function(interaction, df, covariates = NULL) {
  genes <- unique(unlist(str_split(interaction, '[+&_]')))
  
  multiple <- length(unique(df$condition)) > 1

  cols <- c("patient", "OS.time", "OS", "condition", genes, covariates)
  if (!multiple) {
    cols <- setdiff(cols, "condition")
  }
  df <- df %>% select(all_of(cols))

  df <- df %>% mutate(OS.time = OS.time / 365)

  high_expression_group <- data.frame()
  low_expression_group <- data.frame()

  if (multiple) {
    for (condition in unique(df$condition)) {
      tissue_df <- df %>% filter(.data$condition == condition)
      medians <- sapply(genes, function(gene) median(tissue_df[[gene]], na.rm = TRUE))
      
      above_median_flags <- sapply(genes, function(gene) tissue_df[[gene]] > medians[gene])
      below_median_flags <- sapply(genes, function(gene) tissue_df[[gene]] <= medians[gene])
      
      above_all_genes <- rowSums(above_median_flags) == length(genes)
      below_all_genes <- rowSums(below_median_flags) == length(genes)
      
      high_expression_group <- bind_rows(high_expression_group, tissue_df[above_all_genes, ])
      low_expression_group <- bind_rows(low_expression_group, tissue_df[below_all_genes, ])
    }
  } else {
    medians <- sapply(genes, function(gene) median(df[[gene]], na.rm = TRUE))
    above_median_flags <- sapply(genes, function(gene) df[[gene]] > medians[gene])
    below_median_flags <- sapply(genes, function(gene) df[[gene]] <= medians[gene])
    
    above_all_genes <- rowSums(above_median_flags) == length(genes)
    below_all_genes <- rowSums(below_median_flags) == length(genes)
    
    high_expression_group <- df[above_all_genes, ]
    low_expression_group <- df[below_all_genes, ]
  }

  patients_in_both_groups <- intersect(high_expression_group$patient, low_expression_group$patient)
  high_expression_group <- high_expression_group %>% filter(!patient %in% patients_in_both_groups)
  low_expression_group <- low_expression_group %>% filter(!patient %in% patients_in_both_groups)

  high_expression_group <- high_expression_group %>% mutate(group = 1)
  low_expression_group <- low_expression_group %>% mutate(group = 0)

  df <- bind_rows(high_expression_group, low_expression_group)
  df <- df %>% select(-patient)

  high_expression_ids <- paste(rownames(high_expression_group), collapse = ";")
  low_expression_ids <- paste(rownames(low_expression_group), collapse = ";")

  df <- df %>% filter(group %in% c(0, 1))

  n_above_all <- nrow(high_expression_group)
  n_below_all <- nrow(low_expression_group)

  if (n_above_all < MIN_PATIENTS || n_below_all < MIN_PATIENTS) {
    return(list(
      model_full = NA,
      model_reduced = NA,
      n_patients_low = n_below_all,
      n_patients_high = n_above_all
    ))
  }

  df$group <- as.factor(df$group)
  
  # Convert all covariates to factor                               
  #if (!is.null(covariates)) df[covariates] <- lapply(df[covariates], as.factor)
  if (!is.null(covariates)) {
    for (cov in covariates) {
      if (cov == "age_at_diagnosis") {
        df[[cov]] <- as.numeric(df[[cov]])
      } else {
        df[[cov]] <- as.factor(df[[cov]])
      }
    }
  }
                                 
  if (multiple) df$condition <- as.factor(df$condition)

  if (!is.null(covariates) && length(covariates) > 0) {
    valid_covariates <- c()
    for (cov in covariates) {
      if (!cov %in% names(df) || all(is.na(df[[cov]]))) next
      counts <- table(df[[cov]])
      if (all(counts >= MIN_PATIENTS)) valid_covariates <- c(valid_covariates, cov)
    }
    covariates <- if (length(valid_covariates) > 0) valid_covariates else NULL
  }

  # Prepare formulas
  cov_string <- if (!is.null(covariates)) paste(covariates, collapse = " + ") else NULL

  if (multiple) {
    full_rhs <- paste(c("group", cov_string, "strata(condition)"), collapse = " + ")
    reduced_rhs <- paste(c(cov_string, "strata(condition)"), collapse = " + ")
  } else {
    full_rhs <- paste(c("group", cov_string), collapse = " + ")
    reduced_rhs <- cov_string
  }

  formula_full <- as.formula(paste("Surv(OS.time, OS) ~", full_rhs))
  formula_reduced <- if (!is.null(reduced_rhs)) as.formula(paste("Surv(OS.time, OS) ~", reduced_rhs)) else NULL

  result <- tryCatch({
    model_full <- coxph(formula_full, data = df)
    model_reduced <- if (!is.null(formula_reduced)) coxph(formula_reduced, data = df) else NULL

    list(
      model_full = model_full,
      model_reduced = model_reduced,
      n_patients_low = n_below_all,
      n_patients_high = n_above_all
    )
  }, warning = function(w) {
    return(NULL)
  })

  return(result)
}

In [27]:
lrt_sig <- c()
aic_better <- c()
wald_sig <- c()
success_count <- 0

pb <- txtProgressBar(min = 0, max = length(circuits), style = 3)

for (i in seq_along(circuits)) {

  interaction <- circuits[i]

  model_list <- tryCatch(
    {
      survival_analysis(interaction, df,
                        #covariates = c("binary_age", "binary_tumor_stage", "gender"))
                        covariates = c("age_at_diagnosis", "binary_tumor_stage", "gender"))

    },
    error = function(e) {
      message("Error for motif ", interaction, ": ", e$message)
      return(NULL)
    }
  )

  if (is.null(model_list) || is.null(model_list$model_full) || is.null(model_list$model_reduced)) {
    setTxtProgressBar(pb, i)
    next
  }

  success_count <- success_count + 1

  # --- LRT ---
  lrt <- anova(model_list$model_reduced, model_list$model_full, test = "Chisq")
  pval_lrt <- lrt[2, 'Pr(>|Chi|)']
  lrt_sig <- c(lrt_sig, ifelse(pval_lrt < 0.05, 1, 0))

  # --- AIC ---
  aic_full <- AIC(model_list$model_full)
  aic_reduced <- AIC(model_list$model_reduced)
  aic_better <- c(aic_better, ifelse(aic_full < aic_reduced, 1, 0))

  # --- Wald ---
  coef_summary <- summary(model_list$model_full)$coefficients
  if ("group1" %in% rownames(coef_summary)) {
    pval_wald <- coef_summary["group1", "Pr(>|z|)"]
    wald_sig <- c(wald_sig, ifelse(pval_wald < 0.05, 1, 0))
  } else {
    wald_sig <- c(wald_sig, NA)
  }

  setTxtProgressBar(pb, i)
}

close(pb)

cat("Fraction significant by LRT:", mean(lrt_sig, na.rm = TRUE), "\n")
cat("Fraction where full model has lower AIC:", mean(aic_better, na.rm = TRUE), "\n")
cat("Fraction significant by Wald test:", mean(wald_sig, na.rm = TRUE), "\n")

  |======================================================================| 100%
Fraction significant by LRT: 0.9911037 
Fraction where full model has lower AIC: 0.999166 
Fraction significant by Wald test: 0.9902697 


In [30]:
model_list$model_full

Call:
coxph(formula = formula_full, data = df)

                       coef exp(coef) se(coef)      z       p
group1               0.3861    1.4712   0.1422  2.715 0.00663
binary_tumor_stage1  0.5264    1.6927   0.1405  3.746 0.00018
gender1             -0.1864    0.8299   0.1382 -1.348 0.17755

Likelihood ratio test=20.33  on 3 df, p=0.0001449
n= 668, number of events= 300 
   (32 observations deleted due to missingness)

In [31]:
model_list$model_reduced

Call:
coxph(formula = formula_reduced, data = df)

                       coef exp(coef) se(coef)      z        p
binary_tumor_stage1  0.5033    1.6542   0.1397  3.602 0.000315
gender1             -0.1858    0.8305   0.1372 -1.354 0.175718

Likelihood ratio test=13.04  on 2 df, p=0.001472
n= 668, number of events= 300 
   (32 observations deleted due to missingness)

In [9]:
model_list$model_full

Call:
coxph(formula = formula_full, data = df)

                coef exp(coef) se(coef)      z        p
group1       0.74838   2.11358  0.21347  3.506 0.000455
binary_age1  1.58190   4.86419  0.24605  6.429 1.28e-10
gender1     -0.03761   0.96309  0.20829 -0.181 0.856693

Likelihood ratio test=53.42  on 3 df, p=1.49e-11
n= 292, number of events= 114 
   (2 observations deleted due to missingness)

In [10]:
model_list$model_reduced

Call:
coxph(formula = formula_reduced, data = df)

               coef exp(coef) se(coef)      z        p
binary_age1  1.4459    4.2458   0.2376  6.086 1.16e-09
gender1     -0.1788    0.8363   0.2024 -0.883    0.377

Likelihood ratio test=40.4  on 2 df, p=1.691e-09
n= 292, number of events= 114 
   (2 observations deleted due to missingness)

In [105]:
length(cliques)

[1] 4305

In [106]:
length(circuits)

[1] 40186

[1] 0.5650762

In [77]:
model_list

$model_full
Call:
coxph(formula = formula_full, data = df)

                        coef exp(coef) se(coef)      z        p
group1               0.04971   1.05096  0.08634  0.576 0.564809
binary_age1          0.28358   1.32787  0.08052  3.522 0.000428
binary_tumor_stage1  0.79411   2.21248  0.09003  8.821  < 2e-16
gender1             -0.15749   0.85429  0.08977 -1.754 0.079369

Likelihood ratio test=80.95  on 4 df, p=< 2.2e-16
n= 1564, number of events= 632 
   (76 observations deleted due to missingness)

$model_reduced
Call:
coxph(formula = formula_reduced, data = df)

                        coef exp(coef) se(coef)      z        p
binary_age1          0.28682   1.33219  0.08030  3.572 0.000355
binary_tumor_stage1  0.79185   2.20749  0.08995  8.804  < 2e-16
gender1             -0.14882   0.86173  0.08850 -1.682 0.092637

Likelihood ratio test=80.62  on 3 df, p=< 2.2e-16
n= 1564, number of events= 632 
   (76 observations deleted due to missingness)

$n_patients_low
[1] 834

$n_patien

In [67]:
model_list

$model_full
Call:
coxph(formula = formula_full, data = df)

                       coef exp(coef) se(coef)      z        p
group1              -0.2201    0.8025   0.1989 -1.106 0.268641
binary_age1          0.6714    1.9569   0.1982  3.387 0.000706
binary_tumor_stage1  0.7605    2.1393   0.2055  3.701 0.000215

Likelihood ratio test=24.55  on 3 df, p=1.914e-05
n= 783, number of events= 108 
   (15 observations deleted due to missingness)

$model_reduced
Call:
coxph(formula = formula_reduced, data = df)

                      coef exp(coef) se(coef)     z        p
binary_age1         0.6793    1.9724   0.1981 3.428 0.000608
binary_tumor_stage1 0.7606    2.1396   0.2056 3.699 0.000216

Likelihood ratio test=23.32  on 2 df, p=8.618e-06
n= 783, number of events= 108 
   (15 observations deleted due to missingness)

$n_patients_low
[1] 398

$n_patients_high
[1] 400


In [24]:

survival_analysis <- function(interaction, df, covariates = NULL) {
  #cat(interaction, '\n')

  genes <- unique(unlist(str_split(interaction, '[+&_]')))
  
  # Check if multiple conditions are present
  multiple <- length(unique(df$condition)) > 1

  # !!!!!!!!!!!!!!!!
  # Select relevant columns (OS.time, OS, condition, and the genes of interest)
  cols <- c("patient", "OS.time", "OS", "condition", genes, covariates)
  if (!multiple) {
    cols <- setdiff(cols, "condition")
  }
  df <- df %>% select(all_of(cols))

  # Convert OS time to years
  df <- df %>% mutate(OS.time = OS.time / 365)

  # Create empty dataframes for high and low expression groups
  high_expression_group <- data.frame()
  low_expression_group <- data.frame()

  # If multiple tumors are present, split by condition
  #print('Splitting')
  if (multiple) {
    for (condition in unique(df$condition)) {
      tissue_df <- df %>% filter(.data$condition == condition)

      medians <- sapply(genes, function(gene) median(tissue_df[[gene]], na.rm = TRUE))
      
      # Create binary flags for above/below median for each gene
      above_median_flags <- sapply(genes, function(gene) tissue_df[[gene]] > medians[gene])
      below_median_flags <- sapply(genes, function(gene) tissue_df[[gene]] <= medians[gene])
      
      # Identify patients that are above the median for all genes
      above_all_genes <- rowSums(above_median_flags) == length(genes)
      below_all_genes <- rowSums(below_median_flags) == length(genes)
      
      # Split patients into high and low expression groups based on all genes
      high_expression_group <- bind_rows(high_expression_group, tissue_df[above_all_genes, ])
      low_expression_group <- bind_rows(low_expression_group, tissue_df[below_all_genes, ])
    }
  } else {
    # If only one tissue, split into high and low expression groups
    medians <- sapply(genes, function(gene) median(df[[gene]], na.rm = TRUE))
    
    # Create binary flags for above/below median for all genes
    above_median_flags <- sapply(genes, function(gene) df[[gene]] > medians[gene])
    below_median_flags <- sapply(genes, function(gene) df[[gene]] <= medians[gene])
    
    # Identify patients that are above the median for all genes
    above_all_genes <- rowSums(above_median_flags) == length(genes)
    below_all_genes <- rowSums(below_median_flags) == length(genes)
    
    # Split patients into high and low expression groups based on all genes
    high_expression_group <- df[above_all_genes, ]
    low_expression_group <- df[below_all_genes, ]
  }

  # Split into all high vs all low
  patients_in_both_groups <- intersect(high_expression_group$patient, low_expression_group$patient)
  high_expression_group <- high_expression_group %>% filter(!patient %in% patients_in_both_groups)
  low_expression_group <- low_expression_group %>% filter(!patient %in% patients_in_both_groups)

  # Add a 'group' column to indicate high (1) or low (0) expression groups
  high_expression_group <- high_expression_group %>% mutate(group = 1)
  low_expression_group <- low_expression_group %>% mutate(group = 0)

  # Combine the high and low expression groups into one dataframe
  df <- bind_rows(high_expression_group, low_expression_group)
 
  # Patient column is not needed anymore
  df <- df %>% select(-patient)

  # Save the ids (rownames) in each group
  high_expression_ids <- rownames(high_expression_group)
  low_expression_ids <- rownames(low_expression_group)

  # Convert to a ; delimited string of ids
  high_expression_ids <- paste(high_expression_ids, collapse = ";")
  low_expression_ids <- paste(low_expression_ids, collapse = ";")

  # Remove any rows that are not part of the high or low expression groups
  df <- df %>% filter(group %in% c(0, 1))

  # Count the number of patients in each group
  n_above_all <- nrow(high_expression_group)
  n_below_all <- nrow(low_expression_group)

  # If there are too few patients in either group, list of NA values
  if (n_above_all < MIN_PATIENTS || n_below_all < MIN_PATIENTS) {
    return(list(
      hr = NA,
      n_patients_low = n_below_all,
      n_patients_high = n_above_all,
      logrank_pval = NA,
      concordance_index = NA,
      ci_low = NA,
      ci_high = NA,
      se = NA
      #high_expression_ids = high_expression_ids,
      #low_expression_ids = low_expression_ids
    ))
  }
                                 
  # Make sure the group and condition columns are factors

  df$group <- as.factor(df$group)
                                 
  if (!is.null(covariates)) {
    for (cov in covariates) {
      if (cov == "age_at_diagnosis") {
        df[[cov]] <- as.numeric(df[[cov]])
      } else {
        df[[cov]] <- as.factor(df[[cov]])
      }
    }
  }

  if (multiple) {
    df$condition <- as.factor(df$condition)
  }

  # Discard covariates with less than MIN_SAMPLES
  if (!is.null(covariates) && length(covariates) > 0) {
  
    valid_covariates <- c()
  
    for (cov in covariates) {
  
      # skip if column disappeared or is all NA
      if (!cov %in% names(df) || all(is.na(df[[cov]]))) next
  
      counts <- table(df[[cov]])
  
      if (all(counts >= MIN_PATIENTS)) {
        valid_covariates <- c(valid_covariates, cov)
      }
    }
  
    covariates <- valid_covariates
  
    if (length(covariates) == 0) {
      message("No covariates meet minimum sample requirements.")
      covariates <- NULL
    }
  }
                                 
  # Remove all columns that are not needed and run the Cox model
  model_cols <- c("OS.time", "OS", "group", covariates)
  
  if (multiple) {
    model_cols <- c(model_cols, "condition")
  }
  
  df <- df %>% select(all_of(model_cols))

  result <- tryCatch({
      
    cov_string <- if (!is.null(covariates)) paste(covariates, collapse = " + ") else NULL
    
    if (multiple) {
      rhs <- paste(c("group", cov_string, "strata(condition)"), collapse = " + ")
    } else {
      rhs <- paste(c("group", cov_string), collapse = " + ")
    }
    
    formula <- as.formula(paste("Surv(OS.time, OS) ~", rhs))
    
    model <- coxph(formula, data = df)
      
  # !!!!!!!!!!!
  return (model)

    # Extract the metrics
    hr <- exp(coef(model)["group1"])
    logrank_pval <- summary(model)$sctest["pvalue"]
    concordance <- summary(model)$concordance[1]
    ci_low <- exp(confint(model)["group1", 1])
    ci_high <- exp(confint(model)["group1", 2])
    se <- summary(model)$coefficients["group1", "se(coef)"]
    # Propagate error:
    # hr = exp(coef) => err_hr = exp(coef) * se(coef) = hr * se(coef)
    se <- hr * se

    return(list(
      hr = hr,
      n_patients_low = n_below_all,
      n_patients_high = n_above_all,
      logrank_pval = logrank_pval,
      concordance_index = concordance,
      ci_low = ci_low,
      ci_high = ci_high,
      se = se
      #high_expression_ids = high_expression_ids,
      #low_expression_ids = low_expression_ids
    ))
  }, warning = function(w) {
      return(NULL)
  })

  return(result)
}   


In [25]:
# Set significance threshold
alpha <- 0.05

# Vector to store significance results
sig_group <- c()

for (interaction in motifs[0:100]) {
  
  # Wrap the call in tryCatch to handle errors
  model <- tryCatch(
    {
      survival_analysis(interaction, df, covariates = c("binary_age", "binary_tumor_stage", "gender"))
    },
    error = function(e) {
      message("Error for motif ", interaction, ": ", e$message)
      return(NULL)  # return NULL if error occurs
    }
  )
  
  # If the model returned NULL, treat as not significant and continue
  if (is.null(model)) {
    sig_group <- c(sig_group, 0)
    next
  }
  
  # Extract Wald p-values
  wald_p <- summary(model)$coefficients[, "Pr(>|z|)"]
  
  # Check if 'group1' is significant
  sig <- ifelse(wald_p["group1"] < alpha, 1, 0)
  sig_group <- c(sig_group, sig)
}

# Fraction of motifs where group1 is significant
frac_significant <- mean(sig_group)
cat("Fraction of times group1 is significant:", frac_significant, "\n")

Fraction of times group1 is significant: 0.65 


In [24]:
for (interaction in motifs[1:10]) {
    cat(interaction)
    model <- survival_analysis(interaction, df, covariates = c("binary_age", "binary_tumor_stage", "gender"))
    #print(summary(model))
    print(summary(model)$coefficients[, "Pr(>|z|)"])
    cat('\n')
    cat('\n')
    cat('\n')
}



CCL3+CCR1&CCL5+CCR1CCL3+CCR1&CCL5+CCR1 
[1] "Splitting"


Dropping covariate 'gender' due to sparse level(s): 0



             group1         binary_age1 binary_tumor_stage1 
        0.850868441         0.306346978         0.008522297 



CD4+HLA-DQA2&CD4+IL16CD4+HLA-DQA2&CD4+IL16 
[1] "Splitting"


Dropping covariate 'gender' due to sparse level(s): 0



             group1         binary_age1 binary_tumor_stage1 
       4.400056e-01        6.026515e-06        5.067323e-03 



CCL3+CCR1&CCL3+CCR5CCL3+CCR1&CCL3+CCR5 
[1] "Splitting"


...

             group1         binary_age1 binary_tumor_stage1 
        0.065372435         0.001091856         0.028292116 



CD4+HLA-DQA2&CD4+HLA-DPA1CD4+HLA-DQA2&CD4+HLA-DPA1 
[1] "Splitting"


Dropping covariate 'gender' due to sparse level(s): 0



             group1         binary_age1 binary_tumor_stage1 
       2.138550e-01        1.759598e-04        2.173229e-05 



ICAM1+SPN&SIGLEC1+SPNICAM1+SPN&SIGLEC1+SPN 
[1] "Splitting"


Dropping covariate 'gender' due to sparse level(s): 0



             group1         binary_age1 binary_tumor_stage1 
        0.872839551         0.287091650         0.004854194 



CXCL11+CXCR3&CXCL10+CXCR3CXCL11+CXCR3&CXCL10+CXCR3 
[1] "Splitting"
             group1         binary_age1 binary_tumor_stage1             gender1 
       5.196676e-03        5.467841e-04        1.845021e-05        6.292460e-01 





In [12]:
model <- survival_analysis(interaction, df, covariates = c("binary_age", "binary_tumor_stage", "gender"))

reduced <- coxph(
  Surv(OS.time, OS) ~ binary_age + binary_tumor_stage + gender,
  data = df
)

AIC(model, reduced)

CD4+HLA-DQA2&CD4+HLA-DQA1 
[1] "Splitting"


Dropping covariate 'gender' due to sparse level(s): 0

Warning message in AIC.default(model, reduced):
“models are not all fitted to the same number of observations”


,df,AIC
,<dbl>,<dbl>
model,3,862.0952
reduced,3,2110.8432


In [20]:
model

Call:
coxph(formula = formula, data = df)

                       coef exp(coef) se(coef)      z        p
group1              -0.5081    0.6016   0.1818 -2.795 0.005197
binary_age1          0.6115    1.8432   0.1769  3.457 0.000547
binary_tumor_stage1  0.7789    2.1792   0.1819  4.283 1.85e-05
gender1              0.4854    1.6248   1.0054  0.483 0.629246

Likelihood ratio test=38.13  on 4 df, p=1.053e-07
n= 876, number of events= 134 
   (21 observations deleted due to missingness)

group1         binary_age1 binary_tumor_stage1             gender1 
       5.196676e-03        5.467841e-04        1.845021e-05        6.292460e-01

In [10]:
summary(model)$sctest["pvalue"]

test           df       pvalue 
3.293323e+01 3.000000e+00 3.326834e-07

In [44]:
head(df)

,condition,tissue,type,gender,study,OS,OS.time,DSS,DSS.time,DFI,⋯,KIF16B,CYP4F2,TENM1,BATF3,PPP6R1,OR8D4,patient,binary_tumor_stage,age_at_diagnosis,binary_age
,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<int>
TCGA-3C-AAAU-01,Breast Invasive Carcinoma,Breast,Primary Tumor,1,TCGA,0,4047,0,4047,1,⋯,5.9331,-9.965784,-6.5064,0.1257,5.9414,-9.965784,TCGA-3C-AAAU-01,NA,55,0
TCGA-3C-AALI-01,Breast Invasive Carcinoma,Breast,Primary Tumor,1,TCGA,0,4005,0,4005,0,⋯,5.9568,-5.573500,-6.5064,1.8036,6.2196,-9.965784,TCGA-3C-AALI-01,0,50,0
TCGA-3C-AALJ-01,Breast Invasive Carcinoma,Breast,Primary Tumor,1,TCGA,0,1474,0,1474,0,⋯,2.8858,-2.727400,-4.6082,1.8201,5.9286,-9.965784,TCGA-3C-AALJ-01,0,62,1
TCGA-3C-AALK-01,Breast Invasive Carcinoma,Breast,Primary Tumor,1,TCGA,0,1448,0,1448,NA,⋯,5.0211,-3.171400,-3.3076,1.6604,5.2146,-9.965784,TCGA-3C-AALK-01,0,52,0
TCGA-4H-AAAK-01,Breast Invasive Carcinoma,Breast,Primary Tumor,1,TCGA,0,348,0,348,0,⋯,3.8259,-2.388400,-4.0350,1.0915,5.6244,-9.965784,TCGA-4H-AAAK-01,1,50,0
TCGA-5L-AAT0-01,Breast Invasive Carcinoma,Breast,Primary Tumor,1,TCGA,0,1477,0,1477,NA,⋯,4.4337,-5.573500,-4.0350,2.2783,4.9851,-9.965784,TCGA-5L-AAT0-01,0,42,0


In [36]:
summary(model)

                  Length Class  Mode   
hr                1      -none- logical
n_patients_low    1      -none- numeric
n_patients_high   1      -none- numeric
logrank_pval      1      -none- logical
concordance_index 1      -none- logical
ci_low            1      -none- logical
ci_high           1      -none- logical
se                1      -none- logical

In [10]:
## Test all ccc interactions
#ccc_results <- data.frame()
#for (interaction in ccc) {
#  result <- survival_analysis(interaction, df)
#  
#  # Convert the result into a data frame, and add the interaction and type
#  result_df <- data.frame(
#    interaction = interaction,
#    hr = result$hr,
#    n_patients_low = result$n_patients_low,
#    n_patients_high = result$n_patients_high,
#    logrank_pval = result$logrank_pval,
#    concordance_index = result$concordance_index,
#    ci_low = result$ci_low,
#    ci_high = result$ci_high,
#    se = result$se,
#    #high_expression_ids = result$high_expression_ids,
#    #low_expression_ids = result$low_expression_ids,
#    type = "ccc"
#  )
#  
#  # Bind the new row to the results dataframe
#  ccc_results <- bind_rows(ccc_results, result_df)
#}

# Test all crosstalk interactions
crosstalk_results <- data.frame()
for (interaction in head(motifs)) {
  cat(interaction)
  result <- survival_analysis(interaction, df)
  
  # Convert the result into a data frame, and add the interaction and type
  result_df <- data.frame(
    interaction = interaction,
    hr = result$hr,
    n_patients_low = result$n_patients_low,
    n_patients_high = result$n_patients_high,
    logrank_pval = result$logrank_pval,
    concordance_index = result$concordance_index,
    ci_low = result$ci_low,
    ci_high = result$ci_high,
    se=result$se,
    #high_expression_ids = result$high_expression_ids,
    #low_expression_ids = result$low_expression_ids,
    type = "crosstalk"
  )
  
  # Bind the new row to the results dataframe
  crosstalk_results <- bind_rows(crosstalk_results, result_df)
}

# Combine the results
#all_results <- bind_rows(ccc_results, crosstalk_results) %>%
#  arrange(logrank_pval)  # Sorting by logrank pval

# Save results
#tissue_name <- tools::file_path_sans_ext(basename(tissuefile))
#output_file <- file.path(output_dir, paste0(tissue_name, '.csv'))

#cat('Saving results to:', output_file, '\n')
#write_csv(all_results, output_file)

cat('Done: survival.R')


CCL3+CCR1&CCL5+CCR1CCL3+CCR1&CCL5+CCR1 
[1] "Splitting"
CD4+HLA-DQA2&CD4+IL16CD4+HLA-DQA2&CD4+IL16 
[1] "Splitting"
CCL3+CCR1&CCL3+CCR5CCL3+CCR1&CCL3+CCR5 
[1] "Splitting"
CCL5+CCR1&CCL5+CCR5CCL5+CCR1&CCL5+CCR5 
[1] "Splitting"
CD4+HLA-DQA2&CD4+HLA-DRB5CD4+HLA-DQA2&CD4+HLA-DRB5 
[1] "Splitting"
CD4+HLA-DQA2&CD4+HLA-DQA1CD4+HLA-DQA2&CD4+HLA-DQA1 
[1] "Splitting"
Done: survival.R

In [11]:
crosstalk_results

,interaction,hr,n_patients_low,n_patients_high,logrank_pval,concordance_index,ci_low,ci_high,se,type
,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
group1...1,CCL3+CCR1&CCL5+CCR1,0.9745332,311,309,0.90252172,0.5099721,0.6449222,1.4726041,0.2052668,crosstalk
group1...2,CD4+HLA-DQA2&CD4+IL16,0.7930847,346,336,0.24401920,0.5479572,0.5365106,1.1723596,0.1581520,crosstalk
group1...3,CCL3+CCR1&CCL3+CCR5,1.0060168,353,356,0.97483272,0.4989420,0.6930270,1.4603613,0.1912930,crosstalk
group1...4,CCL5+CCR1&CCL5+CCR5,0.7974349,388,388,0.22849883,0.5449266,0.5512944,1.1534716,0.1501855,crosstalk
group1...5,CD4+HLA-DQA2&CD4+HLA-DRB5,0.6747926,267,281,0.08047071,0.5798701,0.4329633,1.0516943,0.1527787,crosstalk
group1...6,CD4+HLA-DQA2&CD4+HLA-DQA1,0.6608663,320,310,0.04837038,0.5641616,0.4368340,0.9997944,0.1395931,crosstalk
